<!--nav--> [🗺 Learning path](README.md) · **36/48** · ◀ [Measuring GPU Code Honestly](./Measuring_GPU_Code_Honestly.ipynb) · [Training Kernels & Memory](./Training_Kernels_And_Memory.ipynb) ▶

# Modern GPU & Model Architecture: Hopper, Blackwell, MLA and Fine-Grained MoE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Modern_GPU_And_Model_Architecture.ipynb)

The kernels in [GPU Architecture & CUDA Kernels](./GPU_Architecture_And_CUDA_Kernels.ipynb)
are written for a machine model that has been stable since Kepler: SMs, warps, shared memory,
a grid of independent blocks. Everything in that notebook still applies. But Hopper and
Blackwell added mechanisms that are not refinements of that model — they change what a fast
kernel looks like — and the model architectures that came after them were designed around the
consequences.

This notebook is about both halves of that, because they are the same story told from two
ends:

- **the silicon** — tensor cores, TMA, thread-block clusters, distributed shared memory,
  FP8 and FP4. Arithmetic got ~70x cheaper in seven years; memory bandwidth got ~9x faster.
- **the models** — MLA and fine-grained MoE, which are what you design when arithmetic is
  nearly free and every byte of KV cache and weight is not.

Two kernels carry the second half, and both run here:
[`11_mla_decode.cu`](https://github.com/sugeerth/gpu-training-notebooks/blob/main/kernels/11_mla_decode.cu)
and
[`12_moe_dispatch.cu`](https://github.com/sugeerth/gpu-training-notebooks/blob/main/kernels/12_moe_dispatch.cu).
No GPU required — they compile with `g++` against the CPU shim.

**One thing this notebook does not do** is hand you a `wgmma` kernel to run. Every kernel in
`kernels/` is verified by CI on a machine with no GPU, and a warp-specialized Hopper kernel
cannot be. Shipping one unverified next to twelve verified ones would quietly undermine the
guarantee the directory makes. So the silicon half explains the mechanisms and quantifies what
they are worth; the kernel half sticks to things that can be checked.

In [ ]:
# Setup. On Colab this clones the repo.
import os, subprocess, sys, math, json, uuid
from pathlib import Path
from IPython.display import HTML, display

def find_repo():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "kernels" / "Makefile").exists():
            return cand
    return None

URL = "https://github.com/sugeerth/gpu-training-notebooks"
BRANCH = "claude/serving-optimization-notebooks-jyerty"   # until this lands on main

REPO = find_repo()
if REPO is None:
    dest = Path("/content/gpu-training-notebooks")
    if not (dest / "kernels" / "Makefile").exists():
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", URL, str(dest)], check=True)
        if not (dest / "kernels" / "Makefile").exists():
            subprocess.run(["git", "fetch", "--depth", "1", "origin", BRANCH], cwd=str(dest))
            subprocess.run(["git", "checkout", "FETCH_HEAD"], cwd=str(dest))
    REPO = dest
KERNELS = REPO / "kernels"

def sh(cmd, cwd=KERNELS, limit=7000):
    """Run a command and show its output, minus the ##KB## line the eval harness reads."""
    p = subprocess.run(cmd, shell=True, cwd=str(cwd), capture_output=True, text=True)
    clean = "\n".join(l for l in (p.stdout or "").splitlines() if not l.startswith("##KB##"))
    out = clean[-limit:]
    if out:
        print(out, end="" if out.endswith("\n") else "\n")
    if p.returncode != 0 and p.stderr:
        print((p.stderr or "")[-2000:], file=sys.stderr)
    return p.returncode

def peek(filename, symbol, before=14):
    lines = (KERNELS / filename).read_text().split("\n")
    start = next((i for i, l in enumerate(lines) if symbol in l and ("(" in l or "=" in l)), None)
    if start is None:
        print(f"{symbol} not found in {filename}"); return
    top = start
    while top > 0 and (lines[top - 1].startswith("//") or lines[top - 1].strip() == ""
                       or lines[top - 1].startswith(("__device__", "__global__", "constexpr"))):
        top -= 1
        if start - top > before * 6:
            break
    end = start
    while end < len(lines) - 1 and lines[end] != "}":
        end += 1
    print("\n".join(lines[top:end + 1]))

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=320):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    display(HTML(f"""
<div id="{div}" style="width:100%;max-width:860px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function go() {{ const DATA = {json.dumps(data)}; const root = d3.select("#{div}"); {js} }}
  if (window.d3) go();
  else {{ const s=document.createElement("script"); s.src="{D3_URL}"; s.onload=go;
          document.head.appendChild(s); }}
}})();
</script>"""))

print("repo :", REPO)
try:
    import torch
    HAVE_GPU = torch.cuda.is_available()
    print("gpu  :", torch.cuda.get_device_name(0) if HAVE_GPU else "none — correctness only")
except ImportError:
    torch, HAVE_GPU = None, False
    print("gpu  : no torch — correctness only")

## Part 1 · What actually changed in the silicon

Four mechanisms, in the order they matter for a kernel author.

### Tensor cores, and then more tensor cores

Since Volta, the arithmetic that matters is not the fp32 SIMT pipeline the kernels in this repo
use — it is a separate unit that executes a small matrix multiply-accumulate as one
instruction. Every generation has widened it and added a narrower format:

| | fp32 SIMT | fp16 tensor | fp8 tensor | fp4 tensor |
|---|---|---|---|---|
| V100 (2017) | 15.7 | 125 | — | — |
| A100 (2020) | 19.5 | 312 | — | — |
| H100 (2022) | 67 | 990 | 1979 | — |
| B200 (2024) | ~80 | ~2250 | ~4500 | ~9000 |

TFLOP/s, dense. The fp32 column moved 5x in seven years. The tensor column moved 18x, and then
the format narrowed twice more on top of that.

**Memory bandwidth over the same period went 900 GB/s → 2039 → 3350 → 8000.** Roughly 9x,
against roughly 70x for arithmetic. That divergence is the single most important fact about
modern GPUs, and the cell below works through what it does to the ridge point.

### TMA — the Tensor Memory Accelerator (Hopper)

Before Hopper, staging a tile from global to shared memory meant every thread computing its own
addresses and issuing its own loads — the `for (int i = tid; i < N; i += blockDim.x)` pattern
in every kernel in `kernels/`. That is a lot of index arithmetic and a lot of registers spent
on addresses rather than data.

TMA replaces it with a **descriptor**: one thread issues a single instruction naming a
multidimensional tile, and a dedicated engine copies it asynchronously, handling the address
generation, the bounds and the swizzling. The warps are free to compute while it runs, and
completion is signalled through a barrier rather than polled.

What it buys is not bandwidth — the bytes are the same. It buys **registers and issue slots**,
which is what a tensor-core kernel is actually short of, and it makes deep software pipelining
practical: stage tile *n+2* while computing on tile *n*.

### Thread block clusters and distributed shared memory (Hopper)

The grid used to be flat: blocks are independent and cannot talk. Hopper adds a level — a
**cluster** of up to 16 blocks co-scheduled on the same GPC, which can read each other's shared
memory directly (DSMEM) and synchronize with each other.

That matters because shared memory per SM stopped growing while tiles kept getting bigger. A
cluster is a way to build a 4x larger tile out of four SMs' shared memory without going back to
HBM for the halo.

### FP8, FP4, and microscaling

Hopper added fp8 (`e4m3`/`e5m2`, covered in
[Training Kernels & Memory](./Training_Kernels_And_Memory.ipynb)). Blackwell adds fp4 and
**microscaling** formats, where the scale factor is not per-tensor or per-row but attached to
every block of 32 values in the format itself.

That is the same argument as the group-wise quantization in `05_dequant_gemv.cu` and the
per-block fp8 scaling in `09_fp8_scaling.cu`, moved into hardware: the finer the group, the
less an outlier costs its neighbours, and at 4 bits you cannot afford to let it cost anything.

In [ ]:
# The divergence, computed rather than asserted. This is the whole argument of this notebook.
GENS = [
    # name      year  fp32   fp16   fp8    fp4   GB/s
    ("V100",    2017, 15.7,  125,   None,  None,  900),
    ("A100",    2020, 19.5,  312,   None,  None, 2039),
    ("H100",    2022, 67.0,  990,   1979,  None, 3350),
    ("B200",    2024, 80.0,  2250,  4500,  9000, 8000),
]

print("Ridge point — FLOP per byte above which a kernel is compute-bound.")
print("Below it, more arithmetic is free and only fewer bytes help.\n")
print(f"{'GPU':<8}{'year':>6}{'GB/s':>7}   {'fp32':>16}{'fp16 tensor':>16}"
      f"{'fp8 tensor':>15}{'fp4 tensor':>14}")
print("-" * 84)
for name, year, f32, f16, f8, f4, bw in GENS:
    cells_ = []
    for tf in (f32, f16, f8, f4):
        cells_.append("        —" if tf is None else f"{tf * 1e12 / (bw * 1e9):9.0f}")
    print(f"{name:<8}{year:>6}{bw:>7}   " + "".join(f"{c:>16}" for c in cells_[:1])
          + "".join(f"{c:>16}" for c in cells_[1:2]) + f"{cells_[2]:>15}{cells_[3]:>14}")

print("""
Two readings, and only one of them is the simple story.

Across generations at a fixed format, the ridge point is not monotonic: fp16 goes
139 -> 153 -> 296 -> 281, rising sharply into Hopper and then falling back slightly on
Blackwell, because B200 added a lot of bandwidth (8 TB/s) alongside the arithmetic.

Within a generation, narrowing the format doubles it every time — H100: 296 fp16 -> 591 fp8.
B200: 281 -> 562 -> 1125. And narrowing the format is exactly what everyone is doing, because
it is also what halves the bytes.

Either way the level is the point, not the trend. A kernel needs *hundreds* of operations per
byte to be compute-bound on any of these machines, and almost nothing in LLM inference does:

  decode GEMV, fp16 weights     1.0 FLOP/byte
  decode attention              0.5
  RMSNorm, SwiGLU               0.25
  prefill GEMM, 512 tokens    ~400

Only the last one clears the bar. Everything a model does while generating a token is one to
three orders of magnitude below it — and got *further* below it with every generation, because
the ceiling rose and the kernels did not move.

That is the pressure the architectures in Parts 2 and 3 are a response to. If arithmetic is
free and bytes are not, you redesign the model to have fewer bytes, and you spend arithmetic
buying that reduction. MLA spends a projection to shrink the KV cache. MoE spends routing to
avoid touching most of the weights. Neither would make sense on a machine where the two were
balanced.""")

In [ ]:
# The same thing as a picture: arithmetic and bandwidth, indexed to V100 = 1.
series = [
    {"name": "tensor-core FLOP/s (best format)",
     "pts": [{"x": g[1], "y": (g[5] or g[4] or g[3]) / 125.0} for g in
             [("V100", 2017, 15.7, 125, None, None, 900),
              ("A100", 2020, 19.5, 312, None, None, 2039),
              ("H100", 2022, 67.0, 990, 1979, None, 3350),
              ("B200", 2024, 80.0, 2250, 4500, 9000, 8000)]]},
    {"name": "memory bandwidth",
     "pts": [{"x": y, "y": bw / 900.0} for y, bw in
             [(2017, 900), (2020, 2039), (2022, 3350), (2024, 8000)]]},
    {"name": "fp32 SIMT FLOP/s",
     "pts": [{"x": y, "y": f / 15.7} for y, f in
             [(2017, 15.7), (2020, 19.5), (2022, 67.0), (2024, 80.0)]]},
]
labels = [{"x": 2017, "t": "V100"}, {"x": 2020, "t": "A100"},
          {"x": 2022, "t": "H100"}, {"x": 2024, "t": "B200"}]

show_d3(r"""
  const W = 820, H = 320, M = {t: 26, r: 190, b: 42, l: 54};
  const svg = root.append("svg").attr("width", W).attr("height", H);
  const x = d3.scaleLinear().domain([2016.6, 2024.4]).range([M.l, W - M.r]);
  const y = d3.scaleLog().domain([0.8, 80]).range([H - M.b, M.t]);
  const col = ["#b45210", "#0a6f78", "#6f8085"];

  svg.append("g").attr("transform", `translate(0,${H - M.b})`)
     .call(d3.axisBottom(x).tickValues([2017,2020,2022,2024]).tickFormat(d3.format("d")));
  svg.append("g").attr("transform", `translate(${M.l},0)`)
     .call(d3.axisLeft(y).tickValues([1,2,5,10,20,50]).tickFormat(d => d + "x"));
  svg.append("text").attr("x", M.l).attr("y", 14).style("font-size","12px")
     .style("fill","#6f8085").text("relative to V100, log scale");

  const line = d3.line().x(d => x(d.x)).y(d => y(d.y));
  DATA.series.forEach((s, i) => {
    svg.append("path").datum(s.pts).attr("fill","none").attr("stroke",col[i])
       .attr("stroke-width",2.2).attr("d",line);
    svg.selectAll(null).data(s.pts).join("circle").attr("cx",d=>x(d.x)).attr("cy",d=>y(d.y))
       .attr("r",3.4).attr("fill",col[i]);
    const last = s.pts[s.pts.length-1];
    svg.append("text").attr("x", x(last.x)+9).attr("y", y(last.y)+4)
       .style("font-size","12px").style("fill",col[i]).text(s.name);
    svg.append("text").attr("x", x(last.x)+9).attr("y", y(last.y)+19)
       .style("font-size","11px").style("fill","#8a8580")
       .style("font-family","ui-monospace,monospace").text(last.y.toFixed(0) + "x since 2017");
  });
  svg.selectAll(null).data(DATA.labels).join("text").attr("x",d=>x(d.x)).attr("y",H-M.b+34)
     .attr("text-anchor","middle").style("font-size","11px").style("fill","#8a8580")
     .style("font-family","ui-monospace,monospace").text(d=>d.t);
""", {"series": series, "labels": labels}, height=340)

print("""The gap between the top line and the middle one is the entire subject. Arithmetic got
about 70x cheaper; bandwidth got about 9x faster. Everything below follows from that ratio.""")

### Why FlashAttention-3 is Hopper-specific even though the algorithm is not

[Attention Kernels From Scratch](./Attention_Kernels_From_Scratch.ipynb) develops
FlashAttention's algorithm — online softmax, tiling — and none of it mentions hardware.
FA-2 runs on Ampere. FA-3 is roughly 1.5–2x faster and runs only on Hopper. The algorithm did
not change; the *scheduling* did, and it uses three of the four mechanisms above:

- **warp specialization** — some warps become producers that only issue TMA copies, others
  consumers that only issue `wgmma`. Instead of every warp alternating between loading and
  computing, the two run concurrently in different warps of the same block.
- **async everything** — TMA copies and `wgmma` are both asynchronous, so the softmax of tile
  *n* overlaps the GEMM of tile *n+1*. On Ampere those two phases serialize.
- **fp8 for the second GEMM**, with the accuracy recovered by keeping the accumulator and the
  softmax statistics in higher precision.

The lesson generalizes: past a certain point kernel performance stops being about arithmetic
and becomes about **keeping several asynchronous engines busy at once**. That is a different
skill from the one `03_sgemm.cu` teaches, and it is why the honest advice at the end of this
notebook is what it is.

## Part 2 · MLA — buying a smaller KV cache with arithmetic

If bytes are expensive and arithmetic is cheap, the KV cache is the obvious target: it is pure
bytes, it grows with context, and at long context it dwarfs the weights.

Every earlier reduction threw information away. GQA shares one K/V head across several query
heads. Sliding windows forget old tokens. Quantization rounds. **MLA stores a low-rank
projection instead**, and reconstructs each head's K and V from it:

$$c_t = W_{DKV} h_t \in \mathbb{R}^{d_c} \qquad K_i = W_{UK}^i c_t \qquad V_i = W_{UV}^i c_t$$

The cache holds `d_c + d_r` floats per token per layer, independent of head count — and every
head still gets its own K and V.

The performance result is the interesting one, and it is where the naive implementation loses.
Decompressing the latent into K and V per token costs a `d_h × d_c` matrix-vector **per token
per head**, which is worse than the traffic it saved. The fix is an identity:

$$q_i \cdot K_i = q_i \cdot (W_{UK}^i c_t) = \left((W_{UK}^i)^\top q_i\right) \cdot c_t = q'_i \cdot c_t$$

Project the *query* once per decode step, and attention runs directly against the cached
latent. `W_UV` absorbs into the output projection the same way.

In [ ]:
sh("make --no-print-directory 11_mla_decode")

Both variants compute the same function, and the check above is against a
decompress-and-attend reference in double — so the absorption identity is **verified rather
than asserted**. A broken absorption is a wrong answer here, not a merely slower one.

Two details that decide whether an MLA implementation is correct:

**RoPE does not absorb.** Rotary embeddings mix position into K in a way that does not commute
with the up-projection — you cannot pull `W_UK` through a position-dependent rotation. DeepSeek
splits the head: a compressed part with no positional encoding, and a small decoupled part
(`d_r = 64`) that carries RoPE and is cached directly. The score is the sum of the two, and
that seam is the part people get wrong.

**MLA is not the smallest cache.** MQA's single shared K/V head is less than half of it. The
point is what each gives up: MQA collapses 32 heads onto one K and one V and pays in quality;
MLA keeps a distinct K and V per head and still lands 3.6x below GQA. The honest comparison is
against GQA, which is the configuration it displaced.

In [ ]:
# KV cache per token per layer, and what that means at context length.
ARCHS = [
    ("MHA  — 32 heads x 128",      2 * 32 * 128 * 2),
    ("GQA  — 8 kv heads x 128",    2 * 8 * 128 * 2),
    ("MQA  — 1 kv head x 128",     2 * 1 * 128 * 2),
    ("MLA  — d_c 512 + d_r 64",    (512 + 64) * 2),
]
LAYERS = 60
POOL_GB = 40.0
print(f"{'architecture':<26}{'B/token/layer':>15}{'@ 32k ctx':>12}{'@ 128k ctx':>13}"
      f"{'seqs in 40 GB @128k':>22}")
print("-" * 90)
seats = {}
for name, b in ARCHS:
    at32 = b * LAYERS * 32768 / 1e9
    at128 = b * LAYERS * 131072 / 1e9
    seats[name.split()[0]] = POOL_GB / at128
    print(f"{name:<26}{b:>15}{at32:>10.1f} GB{at128:>11.1f} GB{POOL_GB/at128:>20.1f}")
print(f"\n({LAYERS} layers, fp16. The last column is how many concurrent conversations fit in a")
print(f" {POOL_GB:.0f} GB KV pool — which is the number that decides your throughput.)")
print(f"""
Against GQA, which is what MLA actually displaced: {seats['GQA']:.1f} concurrent 128k
conversations become {seats['MLA']:.1f} on the same card, with per-head K and V intact rather
than eight query heads sharing one. That is the trade the absorption identity makes affordable.

MQA still fits more ({seats['MQA']:.1f}) and always will — it caches one head instead of 32.
The question is never who caches least; it is what each one gave up to get there.""")

## Part 3 · Fine-grained MoE — and why batching stops helping

The second response to cheap arithmetic and expensive bytes is to stop touching most of the
weights. An MoE layer has `E` experts and routes each token to its top-`k`: 32x the parameters
for the same arithmetic per token.

The headline is true. The part that gets left out is what it does to *memory traffic*, and it
is the opposite of what batching normally buys you.

- A **dense** layer reads its weight matrix **once**, however large the batch is. That is the
  entire reason batching works at decode time.
- An **MoE** layer reads every expert that *any* token in the batch selected. A batch of `B`
  tokens scatters across up to `min(B·k, E)` experts, and each one costs a full weight-matrix
  read even if it received a single token.

So MoE weight traffic **grows with batch size** until every expert is hit, and only then
flattens.

In [ ]:
sh("make --no-print-directory 12_moe_dispatch")

In [ ]:
# The traffic curve, at DeepSeek-V3's shape rather than the toy one.
E, K = 256, 8
D_MODEL, D_FF_EXPERT, D_FF_DENSE = 7168, 2048, 18432
BYTES = 2
expert_bytes = 2 * D_MODEL * D_FF_EXPERT * BYTES          # W1 + W2 for one expert
dense_bytes = 2 * D_MODEL * D_FF_DENSE * BYTES

print(f"DeepSeek-V3 shape: {E} routed experts, top-{K}, d_model {D_MODEL}, "
      f"expert d_ff {D_FF_EXPERT}")
print(f"one expert = {expert_bytes/1e6:.1f} MB   dense equivalent = {dense_bytes/1e6:.1f} MB\n")
print(f"{'batch':>7}{'experts hit':>14}{'MoE weights read':>19}{'vs dense':>11}"
      f"{'per token':>13}")
print("-" * 66)
rows = []
for b in (1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024):
    hit = E * (1.0 - (1.0 - K / E) ** b)     # expected distinct experts, uniform routing
    moe = hit * expert_bytes
    rows.append({"b": b, "hit": hit, "ratio": moe / dense_bytes})
    print(f"{b:>7}{hit:>14.1f}{moe/1e9:>16.2f} GB{moe/dense_bytes:>10.1f}x"
          f"{moe/b/1e6:>11.1f} MB")

print("""
Two readings of that table.

Per *layer*: at batch 1 an MoE reads 8 experts — less than the dense equivalent, which is the
headline result. By batch 64 it reads 200 of 256 and moves 12x what dense would.

Per *token* (last column): this is the number that sets decode speed, and it falls the whole
way. Batching still helps — it just helps far less than it does for a dense model, where the
same column would fall as 1/B all the way down.

Which is why MoE serving looks the way it does:
  * expert parallelism  — spread experts across devices so no single GPU reads them all
  * large batches       — the per-token column keeps falling, so keep going
  * a shared expert     — DeepSeek routes every token through one always-on expert, so some of
                          the traffic is guaranteed reused across the whole batch
  * capacity limits     — cap tokens per expert so one hot expert does not stall the step""")

In [ ]:
show_d3(r"""
  const W = 800, H = 300, M = {t: 24, r: 130, b: 44, l: 62};
  const svg = root.append("svg").attr("width", W).attr("height", H);
  const x = d3.scaleLog().domain([1, 1024]).range([M.l, W - M.r]);
  const y = d3.scaleLinear().domain([0, d3.max(DATA, d => d.ratio) * 1.1]).range([H - M.b, M.t]);
  svg.append("g").attr("transform", `translate(0,${H - M.b})`)
     .call(d3.axisBottom(x).tickValues([1,4,16,64,256,1024]).tickFormat(d3.format("d")));
  svg.append("g").attr("transform", `translate(${M.l},0)`)
     .call(d3.axisLeft(y).ticks(6).tickFormat(d => d + "x"));
  svg.append("line").attr("x1", M.l).attr("x2", W - M.r).attr("y1", y(1)).attr("y2", y(1))
     .attr("stroke", "#6f8085").attr("stroke-dasharray", "4 4");
  svg.append("text").attr("x", W - M.r + 8).attr("y", y(1) + 4).style("font-size", "11.5px")
     .style("fill", "#6f8085").text("dense layer");
  svg.append("path").datum(DATA).attr("fill", "none").attr("stroke", "#b45210")
     .attr("stroke-width", 2.4)
     .attr("d", d3.line().x(d => x(d.b)).y(d => y(d.ratio)));
  svg.selectAll("circle").data(DATA).join("circle").attr("cx", d => x(d.b))
     .attr("cy", d => y(d.ratio)).attr("r", 3.2).attr("fill", "#b45210");
  svg.append("text").attr("x", M.l).attr("y", 13).style("font-size", "12px")
     .style("fill", "#6f8085")
     .text("expert weight bytes read per layer, relative to the dense equivalent");
  svg.append("text").attr("x", (M.l + W - M.r) / 2).attr("y", H - 6)
     .attr("text-anchor", "middle").style("font-size", "12px").style("fill", "#6f8085")
     .text("batch size (tokens in the step)");
""", rows, height=320)

## Part 4 · Putting it together: DeepSeek-V3 as a case study

Every mechanism in this notebook appears in one model, and each one is a response to the same
ratio.

| the choice | what it buys | what it costs |
|---|---|---|
| **MLA**, `d_c` 512 + `d_r` 64 | KV cache 3.6x below GQA with per-head K and V | a query projection per step, and the RoPE seam |
| **256 experts, top-8, fine-grained** | 37B active out of 671B total | weight traffic that grows with batch until every expert is hit |
| **1 shared expert, always on** | some weight traffic guaranteed reused | a little of the sparsity given back |
| **FP8 training** | half the bytes, twice the tensor throughput | per-block scaling, and two silent failure modes |
| **Multi-token prediction** | a draft model for free, so speculation is cheap | an extra head to train |
| **auxiliary-loss-free load balancing** | no gradient fighting the language objective | a bias term to tune |

None of these is a micro-optimization. Each one changes what the model *is* in order to change
how many bytes it moves, which is what you do when arithmetic has gotten 70x cheaper and
bandwidth 9x faster.

The serving consequence, which the rest of this repo works through: a model designed this way
is cheap per token and awkward to serve — it needs expert parallelism, large batches and a
scheduler that understands routing imbalance. See
[Serving Mixture-of-Experts](./MoE_Serving_Expert_Parallelism.ipynb) and
[Long-Context Serving](./LongContext_KV_Compression_Serving.ipynb).

## Part 5 · What you should and should not write by hand

The kernels in this repo are written to be *read*. That is a different goal from being fast,
and it is worth being explicit about where the line is.

**Write by hand:** anything memory-bound. Elementwise ops, normalization, activation functions,
fused epilogues, dequantization, KV-cache gathers, routing and dispatch. These are bandwidth
problems, the arithmetic is trivial, and a straightforward kernel that gets the access pattern
right lands at 85–95% of achievable bandwidth — which is *finished*, because there is nothing
above it. Almost every kernel in `kernels/` is in this category, and so is almost every kernel
a serving engine actually ships.

**Do not write by hand:** GEMM. `03_sgemm.cu` reaches maybe 60–80% of fp32 SIMT peak and is
still ~15x slower than cuBLAS on the same card, because cuBLAS uses tensor cores and it does
not. Closing that gap means `wgmma`, swizzled shared-memory layouts, warp specialization and
software pipelining — and the result would be a worse cuBLAS. Use **CUTLASS/CuTe** when you
need a fused variant nobody ships, **Triton** when you want most of the performance for a tenth
of the code, and the vendor library otherwise.

The value of having written `03_sgemm.cu` is not that you will use it. It is that you now know
what those libraries are doing, why their tile sizes are what they are, and what to look at
when one of them is slow on your shapes.

**And measure before either.** [Measuring GPU Code Honestly](./Measuring_GPU_Code_Honestly.ipynb)
is the other half of this: a kernel at 9% of the ceiling has a bug, a kernel at 92% is done,
and you cannot tell which one you have without a denominator.

## What to take away

1. **Arithmetic got ~70x cheaper; bandwidth got ~9x faster.** Every ridge point in Part 1 moved
   up, and no LLM inference kernel moved with it. That divergence is the reason for everything
   else in this notebook.
2. **Hopper's mechanisms are about scheduling, not arithmetic** — TMA, clusters and async
   `wgmma` exist to keep several engines busy at once. That is why FA-3 is Hopper-only while
   its algorithm is not.
3. **MLA buys a smaller cache with arithmetic**, and the absorption identity is what keeps the
   arithmetic from being charged per cached token. Verified here, not asserted.
4. **MoE inverts the batching argument.** Weight traffic grows with batch until every expert is
   hit; per-token traffic still falls, just far more slowly than dense.
5. **Hand-write the memory-bound kernels and use a library for the GEMMs.** The first category
   is most of what a serving engine runs, and a good implementation is 90% of a ceiling that
   has nothing above it.

### Next

- [Training Kernels & Memory](./Training_Kernels_And_Memory.ipynb) — the training side of the
  same kernels, and where training memory actually goes.
- [Attention Kernels From Scratch](./Attention_Kernels_From_Scratch.ipynb) — the algorithm MLA
  builds on, developed in NumPy.
- [Serving Mixture-of-Experts](./MoE_Serving_Expert_Parallelism.ipynb) — what the MoE traffic
  curve does to a scheduler.
- [Portable Kernels & Precision](./Portable_Kernels_Precision_Matrix.ipynb) — how much of this
  survives a move to AMD.